# Métodos Numéricos
## Semana 6 — Factorización QR
### Gram-Schmidt clásico y Gram-Schmidt modificado

**Docente:** Edwin Fernando Muñoz  
**Programa:** Ciencia de Datos — Universidad Externado de Colombia

En este notebook se construyen numéricamente las matrices \(Q\) y \(R\) a partir de una matriz \(A\).

Se estudian:

1. Gram-Schmidt clásico.
2. Gram-Schmidt modificado.
3. Implementación general del método modificado.


## 1. Matriz de ejemplo

Trabajaremos con

\[
A=
\begin{bmatrix}
1&1\\
1&0\\
0&1
\end{bmatrix}.
\]

Sus columnas son

\[
\mathbf a_1=
\begin{bmatrix}
1\\1\\0
\end{bmatrix},
\qquad
\mathbf a_2=
\begin{bmatrix}
1\\0\\1
\end{bmatrix}.
\]

Buscamos una factorización

\[
\boxed{A=QR}.
\]


In [ ]:
import numpy as np

A = np.array([
    [1., 1.],
    [1., 0.],
    [0., 1.]
])

print("A =")
print(A)


## 2. Gram-Schmidt clásico

Para dos columnas:

\[
r_{11}=\|\mathbf a_1\|_2,
\qquad
\mathbf q_1=\frac{\mathbf a_1}{r_{11}},
\]

\[
r_{12}=\mathbf q_1^T\mathbf a_2,
\]

\[
\mathbf u_2=\mathbf a_2-r_{12}\mathbf q_1,
\]

\[
r_{22}=\|\mathbf u_2\|_2,
\qquad
\mathbf q_2=\frac{\mathbf u_2}{r_{22}}.
\]


In [ ]:
a1 = A[:, 0]
a2 = A[:, 1]

r11 = np.linalg.norm(a1)
q1 = a1 / r11

r12 = q1 @ a2
u2 = a2 - r12 * q1

r22 = np.linalg.norm(u2)
q2 = u2 / r22

Q_clasico = np.column_stack((q1, q2))
R_clasico = np.array([
    [r11, r12],
    [0.0, r22]
])

print("Q =")
print(Q_clasico)

print("\nR =")
print(R_clasico)


### Verificación

Debe cumplirse aproximadamente

\[
A=QR
\]

y

\[
Q^TQ=I.
\]


In [ ]:
print("Q @ R =")
print(Q_clasico @ R_clasico)

print("\nQ.T @ Q =")
print(Q_clasico.T @ Q_clasico)


## 3. Gram-Schmidt modificado

En el método modificado se trabaja con vectores que se actualizan después de cada proyección.

Inicialmente,

\[
\mathbf v_1=\mathbf a_1,
\qquad
\mathbf v_2=\mathbf a_2.
\]

La actualización fundamental es

\[
\boxed{
\mathbf v_j\leftarrow
\mathbf v_j-r_{ij}\mathbf q_i
}.
\]


In [ ]:
v1 = A[:, 0].copy()
v2 = A[:, 1].copy()

r11 = np.linalg.norm(v1)
q1 = v1 / r11

r12 = q1 @ v2
v2 = v2 - r12 * q1

r22 = np.linalg.norm(v2)
q2 = v2 / r22

Q_mod = np.column_stack((q1, q2))
R_mod = np.array([
    [r11, r12],
    [0.0, r22]
])

print("Q =")
print(Q_mod)

print("\nR =")
print(R_mod)


### Verificación del método modificado


In [ ]:
print("Q @ R =")
print(Q_mod @ R_mod)

print("\nQ.T @ Q =")
print(Q_mod.T @ Q_mod)


## 4. Algoritmo general: Gram-Schmidt modificado

Sea

\[
A\in\mathbb{R}^{m\times n},
\qquad m\ge n.
\]

Para cada columna \(i\),

\[
r_{ii}=\|\mathbf v_i\|_2,
\qquad
\mathbf q_i=\frac{\mathbf v_i}{r_{ii}}.
\]

Para cada columna posterior \(j\),

\[
r_{ij}=\mathbf q_i^T\mathbf v_j,
\]

y se actualiza

\[
\mathbf v_j\leftarrow
\mathbf v_j-r_{ij}\mathbf q_i.
\]


In [ ]:
def gram_schmidt_modificado(A):
    A = np.array(A, dtype=float)
    m, n = A.shape

    if m < n:
        raise ValueError("Se requiere m >= n para esta implementación QR reducida.")

    V = A.copy()
    Q = np.zeros((m, n))
    R = np.zeros((n, n))

    for i in range(n):

        R[i, i] = np.linalg.norm(V[:, i])

        if np.isclose(R[i, i], 0.0):
            raise ValueError(
                "Las columnas de A son linealmente dependientes."
            )

        Q[:, i] = V[:, i] / R[i, i]

        for j in range(i + 1, n):
            R[i, j] = Q[:, i] @ V[:, j]
            V[:, j] = V[:, j] - R[i, j] * Q[:, i]

    return Q, R


## 5. Aplicación del algoritmo general al ejemplo


In [ ]:
Q, R = gram_schmidt_modificado(A)

print("Q =")
print(Q)

print("\nR =")
print(R)

print("\nQ @ R =")
print(Q @ R)

print("\nQ.T @ Q =")
print(Q.T @ Q)


## 6. Probar otra matriz

Ahora probamos el algoritmo con

\[
A=
\begin{bmatrix}
1&1\\
1&2\\
1&3\\
1&4
\end{bmatrix}.
\]


In [ ]:
A2 = np.array([
    [1., 1.],
    [1., 2.],
    [1., 3.],
    [1., 4.]
])

Q2, R2 = gram_schmidt_modificado(A2)

print("Q =")
print(Q2)

print("\nR =")
print(R2)

print("\nVerificación A ≈ Q @ R:")
print(Q2 @ R2)


## 7. Comparación con NumPy

NumPy permite calcular directamente una factorización QR con `np.linalg.qr`.

Los signos de algunas columnas de \(Q\) y de las filas correspondientes de \(R\) pueden cambiar y, aun así, la factorización sigue siendo válida.


In [ ]:
Q_np, R_np = np.linalg.qr(A2, mode='reduced')

print("Q calculada por NumPy =")
print(Q_np)

print("\nR calculada por NumPy =")
print(R_np)

print("\nVerificación =")
print(Q_np @ R_np)


## Síntesis

El método de Gram-Schmidt modificado construye

\[
\boxed{A=QR}
\]

actualizando cada columna después de calcular una proyección.

La matriz \(Q\) tiene columnas ortonormales y \(R\) es triangular superior.

Esta estructura será utilizada posteriormente para resolver mínimos cuadrados mediante

\[
\boxed{
R\mathbf{x}=Q^T\mathbf b
}.
\]
